In [1]:

!pip install matplotlib seaborn scikit-learn xgboost genomic-benchmarks

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 21.6 MB/s eta 0:00:0000:0100:01
  Created wheel for genomic-benchmarks: filename=genomic_benchmarks-1.0.0-py3-none-any.whl size=22550 sha256=02593af67ba23edc2ed189c7acaf3c6c7ad3ca7289a652b7556e714b69de4924
  Stored in directory: /root/.cache/pip/wheels/a5/a1/e0/b3f522520ac6dbd26a92b9840e06d4ae34ec79c1f0a8391d74
Successfully built genomic-benchmarks


In [2]:
# Import Required Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# Machine Learning Libraries
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.feature_extraction.text import CountVectorizer
import xgboost as xgb

# Genomic Benchmark Dataset
from genomic_benchmarks.dataset_getters.pytorch_datasets import get_dataset

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
# Load and Explore Dataset
print("Loading genomic benchmark dataset...")

# Load the genomic benchmark dataset
train_dset = get_dataset("demo_coding_vs_intergenomic_seqs", split="train", version=0)
test_dset = get_dataset("demo_coding_vs_intergenomic_seqs", split="test", version=0)

# Extract sequences and labels
train_seqs = [item[0] for item in train_dset]
train_labels = [item[1] for item in train_dset]
test_seqs = [item[0] for item in test_dset]
test_labels = [item[1] for item in test_dset]

print(f"Training set size: {len(train_seqs)}")
print(f"Test set size: {len(test_seqs)}")
print(f"Sample sequence type: {type(train_seqs[0])}")
print(f"Sample label type: {type(train_labels[0])}")

Loading genomic benchmark dataset...
Training set size: 75000
Test set size: 25000
Sample sequence type: <class 'str'>
Sample label type: <class 'int'>


In [4]:
# Explore Dataset Structure
print("=== Dataset Exploration ===")

# Check sequence lengths
train_seq_lengths = [len(seq) for seq in train_seqs]
test_seq_lengths = [len(seq) for seq in test_seqs]

print(f"Training sequence lengths - Min: {min(train_seq_lengths)}, Max: {max(train_seq_lengths)}, Mean: {np.mean(train_seq_lengths):.2f}")
print(f"Test sequence lengths - Min: {min(test_seq_lengths)}, Max: {max(test_seq_lengths)}, Mean: {np.mean(test_seq_lengths):.2f}")

# Check class distribution
train_label_counts = Counter(train_labels)
test_label_counts = Counter(test_labels)

print(f"\nTraining set class distribution: {dict(train_label_counts)}")
print(f"Test set class distribution: {dict(test_label_counts)}")

# Show sample sequences
print(f"\nSample sequences:")
for i in range(3):
    print(f"Sequence {i+1} (Label {train_labels[i]}): {train_seqs[i][:50]}...")

=== Dataset Exploration ===
Training sequence lengths - Min: 200, Max: 200, Mean: 200.00
Test sequence lengths - Min: 200, Max: 200, Mean: 200.00

Training set class distribution: {0: 37500, 1: 37500}
Test set class distribution: {0: 12500, 1: 12500}

Sample sequences:
Sequence 1 (Label 0): ATGAGAAACTTTGAATAATAGAGATTGAACATGTGAGAACAGCTACCTAG...
Sequence 2 (Label 0): GCTGCCTTTCCGCCGTTCGATTCCCACTTCCTTCAGAAGGGCGCACTCTT...
Sequence 3 (Label 0): CCGGCGCTCGGACGGACTGACTTGCTGACCGCCCGCCGGAGGCACACCCC...


In [5]:
# DNA Sequence Preprocessing
def preprocess_sequences(sequences, max_length=None):
    """
    Preprocess DNA sequences by cleaning and standardizing
    """
    processed_seqs = []
    
    for seq in sequences:
        # Convert to uppercase
        seq = seq.upper()
        
        # Replace unknown nucleotides with 'N'
        valid_nucleotides = set(['A', 'T', 'G', 'C', 'N'])
        seq = ''.join([base if base in valid_nucleotides else 'N' for base in seq])
        
        processed_seqs.append(seq)
    
    # Determine max length if not provided
    if max_length is None:
        max_length = max(len(seq) for seq in processed_seqs)
    
    # Pad or truncate sequences to uniform length
    uniform_seqs = []
    for seq in processed_seqs:
        if len(seq) < max_length:
            # Pad with 'N'
            seq = seq + 'N' * (max_length - len(seq))
        elif len(seq) > max_length:
            # Truncate
            seq = seq[:max_length]
        uniform_seqs.append(seq)
    
    return uniform_seqs, max_length

# Preprocess sequences
print("Preprocessing sequences...")
train_seqs_processed, max_len = preprocess_sequences(train_seqs)
test_seqs_processed, _ = preprocess_sequences(test_seqs, max_length=max_len)

print(f"Processed sequences to uniform length: {max_len}")
print(f"Sample processed sequence: {train_seqs_processed[0][:50]}...")

Preprocessing sequences...
Processed sequences to uniform length: 200
Sample processed sequence: ATGAGAAACTTTGAATAATAGAGATTGAACATGTGAGAACAGCTACCTAG...


In [6]:
# Feature Engineering
def nucleotide_composition(sequence):
    """Calculate nucleotide composition features"""
    seq_len = len(sequence)
    composition = {
        'A_count': sequence.count('A') / seq_len,
        'T_count': sequence.count('T') / seq_len,
        'G_count': sequence.count('G') / seq_len,
        'C_count': sequence.count('C') / seq_len,
        'N_count': sequence.count('N') / seq_len,
        'GC_content': (sequence.count('G') + sequence.count('C')) / seq_len,
        'AT_content': (sequence.count('A') + sequence.count('T')) / seq_len
    }
    return composition

def generate_kmers(sequence, k=5):
    """Generate k-mers from sequence"""
    return [sequence[i:i+k] for i in range(len(sequence)-k+1)]

def create_kmer_features(sequences, k=5, max_features=1000):
    """Create k-mer frequency features"""
    # Generate k-mer strings for CountVectorizer
    kmer_seqs = []
    for seq in sequences:
        kmers = generate_kmers(seq, k)
        kmer_seqs.append(' '.join(kmers))
    
    # Use CountVectorizer to create feature matrix
    vectorizer = CountVectorizer(max_features=max_features, token_pattern=r'\b\w+\b')
    kmer_matrix = vectorizer.fit_transform(kmer_seqs)
    
    return kmer_matrix.toarray(), vectorizer

print("Creating features...")

# Create composition features
train_composition = [nucleotide_composition(seq) for seq in train_seqs_processed]
test_composition = [nucleotide_composition(seq) for seq in test_seqs_processed]

train_comp_df = pd.DataFrame(train_composition)
test_comp_df = pd.DataFrame(test_composition)

# Create k-mer features (using 5-mers)
train_kmer_features, kmer_vectorizer = create_kmer_features(train_seqs_processed, k=5, max_features=500)
test_kmer_features = kmer_vectorizer.transform([' '.join(generate_kmers(seq, 5)) for seq in test_seqs_processed]).toarray()

# Combine composition and k-mer features
X_train = np.hstack([train_comp_df.values, train_kmer_features])
X_test = np.hstack([test_comp_df.values, test_kmer_features])

print(f"Training feature matrix shape: {X_train.shape}")
print(f"Test feature matrix shape: {X_test.shape}")
print(f"Feature types: {train_comp_df.columns.tolist()[:7]} + {train_kmer_features.shape[1]} k-mer features")

Creating features...
Training feature matrix shape: (75000, 507)
Test feature matrix shape: (25000, 507)
Feature types: ['A_count', 'T_count', 'G_count', 'C_count', 'N_count', 'GC_content', 'AT_content'] + 500 k-mer features


In [7]:
# Data Splitting and Preparation
# Convert labels to numpy arrays
y_train = np.array(train_labels)
y_test = np.array(test_labels)

# Create validation split from training data
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Scale features for SVC and Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_split)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Data splitting and scaling completed!")
print(f"Training set: {X_train_split.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")
print(f"Class distribution in training: {Counter(y_train_split)}")
print(f"Class distribution in validation: {Counter(y_val)}")

Data splitting and scaling completed!
Training set: (60000, 507)
Validation set: (15000, 507)
Test set: (25000, 507)
Class distribution in training: Counter({0: 30000, 1: 30000})
Class distribution in validation: Counter({1: 7500, 0: 7500})


In [10]:
# Add this at the top of your notebook or before using the metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


In [11]:
# XGBoost Model Training and Evaluation
print("=== XGBoost Model Training ===")

# Define XGBoost parameters for grid search
xgb_params = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0]
}

# Initialize XGBoost classifier
xgb_model = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

# Grid search with cross-validation
print("Performing grid search for XGBoost...")
xgb_grid = GridSearchCV(
    xgb_model, xgb_params, cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)

# Train the model
xgb_grid.fit(X_train_split, y_train_split)

# Best model
best_xgb = xgb_grid.best_estimator_
print(f"Best XGBoost parameters: {xgb_grid.best_params_}")

# Evaluate on validation set
xgb_test_pred = best_xgb.predict(X_test)
xgb_test_accuracy = accuracy_score(y_test, xgb_test_pred)
xgb_test_precision = precision_score(y_test, xgb_test_pred, average='weighted')
xgb_test_recall = recall_score(y_test, xgb_test_pred, average='weighted')
xgb_test_f1 = f1_score(y_test, xgb_test_pred, average='weighted')

print(f"XGBoost Test Accuracy: {xgb_test_accuracy:.4f}")
print(f"XGBoost Test Precision: {xgb_test_precision:.4f}")
print(f"XGBoost Test Recall: {xgb_test_recall:.4f}")
print(f"XGBoost Test F1-score: {xgb_test_f1:.4f}")

print("\nXGBoost Classification Report:")
print(classification_report(y_test, xgb_test_pred))

=== XGBoost Model Training ===
Performing grid search for XGBoost...
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best XGBoost parameters: {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 200, 'subsample': 1.0}
XGBoost Test Accuracy: 0.8846
XGBoost Test Precision: 0.8847
XGBoost Test Recall: 0.8846
XGBoost Test F1-score: 0.8846

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.89      0.89     12500
           1       0.89      0.88      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000



In [12]:
# Logistic Regression Training and Evaluation
print("=== Logistic Regression Training ===")

# Define Logistic Regression parameters for grid search
lr_params = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

# Initialize Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)

# Grid search with cross-validation
print("Performing grid search for Logistic Regression...")
lr_grid = GridSearchCV(
    lr_model, lr_params, cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)

# Train the model using scaled features
lr_grid.fit(X_train_scaled, y_train_split)

# Best model
best_lr = lr_grid.best_estimator_
print(f"Best Logistic Regression parameters: {lr_grid.best_params_}")

# Evaluate on validation set
lr_val_pred = best_lr.predict(X_val_scaled)
lr_val_accuracy = accuracy_score(y_val, lr_val_pred)
print(f"Logistic Regression Validation Accuracy: {lr_val_accuracy:.4f}")

# Evaluate on test set
lr_test_pred = best_lr.predict(X_test_scaled)
lr_test_accuracy = accuracy_score(y_test, lr_test_pred)
lr_test_precision = precision_score(y_test, lr_test_pred, average='weighted')
lr_test_recall = recall_score(y_test, lr_test_pred, average='weighted')
lr_test_f1 = f1_score(y_test, lr_test_pred, average='weighted')

print(f"Logistic Regression Test Accuracy: {lr_test_accuracy:.4f}")
print(f"Logistic Regression Test Precision: {lr_test_precision:.4f}")
print(f"Logistic Regression Test Recall: {lr_test_recall:.4f}")
print(f"Logistic Regression Test F1-score: {lr_test_f1:.4f}")

print("\nLogistic Regression Classification Report:")
print(classification_report(y_test, lr_test_pred))

=== Logistic Regression Training ===
Performing grid search for Logistic Regression...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Logistic Regression parameters: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}
Logistic Regression Validation Accuracy: 0.8813
Logistic Regression Test Accuracy: 0.8815
Logistic Regression Test Precision: 0.8815
Logistic Regression Test Recall: 0.8815
Logistic Regression Test F1-score: 0.8815

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.88      0.88     12500
           1       0.88      0.89      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000



In [13]:
# Naive Bayes
from sklearn.naive_bayes import GaussianNB

print("=== Naive Bayes Model Training ===")

nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train_split)

nb_val_pred = nb_model.predict(X_val_scaled)
nb_test_pred = nb_model.predict(X_test_scaled)
nb_test_accuracy = accuracy_score(y_test, nb_test_pred)
nb_test_precision = precision_score(y_test, nb_test_pred, average='weighted')
nb_test_recall = recall_score(y_test, nb_test_pred, average='weighted')
nb_test_f1 = f1_score(y_test, nb_test_pred, average='weighted')

print(f"Naive Bayes Test Accuracy: {nb_test_accuracy:.4f}")
print(f"Naive Bayes Test Precision: {nb_test_precision:.4f}")
print(f"Naive Bayes Test Recall: {nb_test_recall:.4f}")
print(f"Naive Bayes Test F1-score: {nb_test_f1:.4f}")

print("\nNaive Bayes Classification Report:")
print(classification_report(y_test, nb_test_pred))

=== Naive Bayes Model Training ===
Naive Bayes Test Accuracy: 0.7799
Naive Bayes Test Precision: 0.7800
Naive Bayes Test Recall: 0.7799
Naive Bayes Test F1-score: 0.7799

Naive Bayes Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.77      0.78     12500
           1       0.78      0.79      0.78     12500

    accuracy                           0.78     25000
   macro avg       0.78      0.78      0.78     25000
weighted avg       0.78      0.78      0.78     25000

